In [ ]:
import importlib
import numpy as np
from pyntcloud import PyntCloud
import functions
import pandas as pd

importlib.reload(functions)

In [11]:
###############################################################
#
#          PART 1: PREPROCESSING AND CALIBRATION
#
###############################################################

# Paths to the calibration files and to one of the sample videos
K_PATH = "LaserScanner_project_data/calibration/K.txt"
DIST_PATH = "LaserScanner_project_data/calibration/dist.txt"
VIDEO_PATH = "LaserScanner_project_data/data/cup1.mp4"

# Load intrinsic matrix (3x3) and distortion coefficients
K = np.loadtxt(K_PATH)
dist = np.loadtxt(DIST_PATH)



In [ ]:
# INITIALIZATION
MIN_POINTS_FOR_PLANE = 10 
W, H = 0.25, 0.15 
K_INV = np.linalg.inv(K)
OBJECT_POINTS = np.array([
    [0, 0, 0],
    [W, 0, 0],
    [W, H, 0],
    [0, H, 0],
], dtype=np.float32)
all_points = []
all_colors = []
P1 = None
P2 = None
rect1 = None
rect2 = None

for frame in functions.read_undistorted_frames(VIDEO_PATH, K, dist):
    
    # The camera is fixed all time so calculate plane only once
    if P1 is None or P2 is None:
        
        candidates   = functions.find_rectangles(frame)
        rect1, rect2 = functions.assign_rectangles(candidates)

        if rect1 is None or rect2 is None:
            print(f"Warning: Could not detect marker planes")
            continue  

        P1 = functions.estimate_plane_from_rectangle(OBJECT_POINTS, rect1, K)
        P2 = functions.estimate_plane_from_rectangle(OBJECT_POINTS, rect2, K)

    laser_points = functions.detect_laser_points(frame)
    
    if laser_points.shape[0] == 0:
        continue

    pts_rect1, pts_rect2, pts_object = functions.classify_laser_points(laser_points, rect1, rect2)

    # Backproject pixel coordinates to 3D direction vectors
    dirs_rect1   = functions.backproject_points(pts_rect1, K_INV)
    dirs_rect2   = functions.backproject_points(pts_rect2, K_INV)
    dirs_object  = functions.backproject_points(pts_object, K_INV)

    # Intersect laser rays with target marker planes P1 and P2
    q_rect1, _   = functions.intersect_rays_with_plane(dirs_rect1, *P1)
    q_rect2, _   = functions.intersect_rays_with_plane(dirs_rect2, *P2)

    q_calib = (
        np.vstack([q_rect1, q_rect2])
        if (q_rect1.size and q_rect2.size)
        else (q_rect1 if q_rect1.size else q_rect2)
    )

    if q_calib.shape[0] < MIN_POINTS_FOR_PLANE:
        continue

    # Fit current 3D laser plane PL
    PL_point, PL_normal = functions.fit_plane(q_calib)
    if PL_point is None:
        continue

    # Reconstruct 3D points on the target object
    object_points_3d, valid_mask = functions.intersect_rays_with_plane(dirs_object, PL_point, PL_normal)
    object_colors = functions.get_pixel_colors(pts_object[valid_mask], frame)

    if object_points_3d.shape[0] > 0:
        all_points.append(object_points_3d)
        all_colors.append(object_colors)

# Construct PLY format points
if len(all_points) > 0:
    point_cloud = np.vstack(all_points)
    point_cloud_colors = np.vstack(all_colors)

    df = pd.DataFrame(point_cloud, columns=["x", "y", "z"])
    df["red"] = point_cloud_colors[:, 0]
    df["green"] = point_cloud_colors[:, 1]
    df["blue"] = point_cloud_colors[:, 2]

    cloud = PyntCloud(df)
    cloud.to_file("output1.ply")
    print(
        f"Reconstruction completed successfully: {len(point_cloud)} points saved to output.ply."
    )
else:
    print("Reconstruction failed: No 3D object points detected.")